# MetaCog-Triage v2 Run (Kaggle)

**Before running:** upload the `metacog_triage_release` folder as a Kaggle *Dataset* (after running `prepare_release.py` locally so `tasks/` is populated), attach it to this notebook, and enable a GPU (T4 is enough).

Set `PKG_INPUT` below to the attached dataset path.

In [ ]:
PKG_INPUT = '/kaggle/input/metacog-triage-release'  # adjust to your dataset slug
MODELS = ['qwen', 'smollm', 'granite', 'tinyllama']  # add 'qwen3b', 'phi' if VRAM allows

import shutil, os
shutil.copytree(PKG_INPUT, '/kaggle/working/pkg', dirs_exist_ok=True)
os.chdir('/kaggle/working/pkg')
print(os.listdir('.'))

In [ ]:
%pip install -q transformers accelerate sentencepiece

In [ ]:
# Generate + validate v2 tasks if not already present in the dataset
import os, subprocess, sys
if not os.path.exists('tasks/metacog_tasks_v2.jsonl'):
    subprocess.run([sys.executable, 'tasks/generate_tasks_v2.py'], check=True)
subprocess.run([sys.executable, 'tasks/validate_tasks.py', 'tasks/metacog_tasks_v2.jsonl', '--expect-v2'], check=True)

In [ ]:
# Baselines on v2 (fast, CPU)
import subprocess, sys
subprocess.run([sys.executable, 'baselines/baselines.py',
                '--tasks', 'tasks/metacog_tasks_v2.jsonl',
                '--output', 'analysis/baseline_results_v2.csv'], check=True)

In [ ]:
# Main run: ~1-2h for 4 models x 200 tasks on a T4
import subprocess, sys
subprocess.run([sys.executable, 'src/run_benchmark.py',
                '--models', *MODELS,
                '--tasks', 'tasks/metacog_tasks_v2.jsonl',
                '--output', 'results/v2_run1/'], check=True)

In [ ]:
# ALIGNMENT-ARTIFACT HYPOTHESIS TEST (research_designs/HYPOTHESIS_alignment_artifact.md)
# Base (non-instruct) checkpoints of the same families, same 200 tasks.
# Pre-registered prediction: MORE bluffing, LESS over-escalation collapse than instruct twins.
# Compare results/hypothesis_run/comparison_table.md against results/v2_run1/.
import subprocess, sys
subprocess.run([sys.executable, 'src/run_benchmark.py',
                '--models', 'qwen_base', 'smollm_base',
                '--tasks', 'tasks/metacog_tasks_v2.jsonl',
                '--output', 'results/hypothesis_run/'], check=True)

In [ ]:
# Calibration recompute on the new v2 results
import subprocess, sys, glob
subprocess.run([sys.executable, 'baselines/recompute_calibration.py',
                *sorted(glob.glob('results/v2_run1/*_results.json'))], check=True)

In [ ]:
# Bundle outputs for download (Kaggle 'Output' tab)
import shutil
shutil.make_archive('/kaggle/working/v2_results_bundle', 'zip', 'results/v2_run1')
shutil.copy('analysis/baseline_results_v2.csv', '/kaggle/working/')
print('Done. Download v2_results_bundle.zip and baseline_results_v2.csv from the notebook Output tab.')

## After downloading
1. Copy `v2_run1/` into `metacog_triage_release/results/` locally.
2. Check the **By Variant** section of `comparison_table.md` — standard-vs-control gap is the headline number.
3. Compare baselines against the pre-registered predictions in `analysis/baseline_summary.md`.
4. Update paper §5 and re-run the push script.